In [1]:
# Load tests from results folder
import pickle, os
from tqdm import tqdm

def load_tests(symbol, conf):
    folder = f'/_data/base/{symbol}/results/{conf}'
    assert os.path.isdir(folder), f'Results folder not found: {folder}'

    files = [fn for fn in os.listdir(folder) if fn.endswith('.pkl')]
    files.sort()
    assert len(files) > 0, f'No result files in: {folder}'

    sizes_mb = 0.0
    count = 0

    test_data: list = []
    pbar = tqdm(total=len(files), desc=f'Loading tests for {symbol} {conf}')

    for fn in files:
        pbar.update(1)
        path = f'{folder}/{fn}'
        assert os.path.isfile(path), f'Not a file: {path}'
        size_mb = os.path.getsize(path) / 1024 / 1024
        with open(path, 'rb') as f:
            data = pickle.load(f)
        assert isinstance(data, list), f'File is not a list: {path}'
        test_data.extend(data)
        sizes_mb += size_mb
        count += len(data)
    
    pbar.close()

    print(f'Total size: {sizes_mb:.3f} MB')
    print(f'Loaded total: {len(test_data)} tests from {len(files)} files')

    return test_data

In [2]:
# Load confs
import json

def load_confs():
    folder = './run_confs'
    confs = {}

    for file in os.listdir(folder):
        if not file.endswith('.json'):
            continue

        with open(f'{folder}/{file}', 'r') as f:
            conf = json.load(f)
            confs[file.replace('.json', '')] = conf

    return confs

In [3]:
# Base parameters for cost regression
import numpy as np

# def get_base_metrics(params, volume_usd):
#     """
#     Метрики:
#     - average_pnl_pct: средняя прибыль в процентах
#     - average_position_duration_s: средняя длительность позиции в секундах
#     - positions_per_year: количество позиций в год
#     - cost_per_year_pct: стоимость в процентах в год
#     - cost_per_year_usd: стоимость в USD в год
#     - avg_chases_per_position: среднее количество чейзов в позиции
#     - avg_turnover_per_usd: средний оборот на доллар в год
#     - avg_turnover_per_year: средний оборот в год
#     """
#     metrics = {}

#     # Average PnL% per position
#     base_average_pnl_pct = np.mean(params[:, 0])
#     metrics['average_pnl_pct'] = base_average_pnl_pct

#     # Average position duration (seconds)
#     base_average_position_duration_s = np.mean(params[:, 1])
#     metrics['average_position_duration_s'] = base_average_position_duration_s

#     # Positions per year
#     base_positions_per_year = 365*24*60*60.0 / base_average_position_duration_s
#     metrics['positions_per_year'] = base_positions_per_year

#     # Cost per year
#     base_cost_per_year_pct = base_average_pnl_pct * 365*24*60*60.0 / base_average_position_duration_s
#     metrics['cost_per_year_pct'] = base_cost_per_year_pct

#     # Cost per year in USD
#     base_cost_per_year_usd = base_cost_per_year_pct * volume_usd / 100
#     metrics['cost_per_year_usd'] = base_cost_per_year_usd

#     # Average chases per position
#     base_avg_chases_per_position = np.mean(params[:, 2])
#     metrics['avg_chases_per_position'] = base_avg_chases_per_position

#     # Annual turnover per dollar per year
#     base_avg_turnover_per_usd = base_positions_per_year * base_avg_chases_per_position
#     metrics['avg_turnover_per_usd'] = base_avg_turnover_per_usd

#     # And in USDT
#     base_avg_turnover_per_year = base_avg_turnover_per_usd * volume_usd
#     metrics['avg_turnover_per_year'] = base_avg_turnover_per_year

#     return metrics

def get_base_metrics(params, volume_usd):
    """
    Метрики (с раздельной декомпозицией по знаку PnL и регрессией по долям):
    - average_pnl_pct: ожидаемый средний PnL за позицию (%), как смесь (+/-/0) по долям
    - average_position_duration_s: ожидаемая средняя длительность позиции (с), смесь по долям
    - positions_per_year: позиций/год = секунды_в_году / average_position_duration_s
    - cost_per_year_pct: годовая "стоимость" (%) = average_pnl_pct * positions_per_year
    - cost_per_year_usd: в USD
    - avg_chases_per_position: ожидаемое среднее число чейзов (смесь по долям)
    - avg_turnover_per_usd: оборот на доллар/год = positions_per_year * avg_chases_per_position
    - avg_turnover_per_year: оборот в год = avg_turnover_per_usd * volume_usd

    Доп. поля по декомпозиции:
    - avg_pos_pnl_pct, avg_neg_pnl_pct
    - positive_share, negative_share, zero_share
    - avg_pos_duration_s, avg_neg_duration_s, avg_zero_duration_s
    - avg_chases_pos, avg_chases_neg, avg_chases_zero
    """
    metrics = {}

    if params is None or len(params) == 0:
        return {
            'average_pnl_pct': 0.0,
            'average_position_duration_s': 0.0,
            'positions_per_year': 0.0,
            'cost_per_year_pct': 0.0,
            'cost_per_year_usd': 0.0,
            'avg_chases_per_position': 0.0,
            'avg_turnover_per_usd': 0.0,
            'avg_turnover_per_year': 0.0,
            'avg_pos_pnl_pct': 0.0,
            'avg_neg_pnl_pct': 0.0,
            'positive_share': 0.0,
            'negative_share': 0.0,
            'zero_share': 0.0,
            'avg_pos_duration_s': 0.0,
            'avg_neg_duration_s': 0.0,
            'avg_zero_duration_s': 0.0,
            'avg_chases_pos': 0.0,
            'avg_chases_neg': 0.0,
            'avg_chases_zero': 0.0,
        }

    pnl = params[:, 0].astype(np.float64)
    dur = params[:, 1].astype(np.float64)
    chs = params[:, 2].astype(np.float64)

    pos_mask = pnl > 0
    neg_mask = pnl < 0
    zero_mask = pnl == 0

    n = float(len(pnl))
    n_pos = float(np.sum(pos_mask))
    n_neg = float(np.sum(neg_mask))
    n_zero = float(np.sum(zero_mask))

    p_pos = n_pos / n if n > 0 else 0.0
    p_neg = n_neg / n if n > 0 else 0.0
    p_zero = n_zero / n if n > 0 else 0.0

    avg_pos_pnl = float(np.mean(pnl[pos_mask])) if n_pos > 0 else 0.0
    avg_neg_pnl = float(np.mean(pnl[neg_mask])) if n_neg > 0 else 0.0
    avg_zero_pnl = 0.0  # по определению

    avg_pos_dur = float(np.mean(dur[pos_mask])) if n_pos > 0 else 0.0
    avg_neg_dur = float(np.mean(dur[neg_mask])) if n_neg > 0 else 0.0
    avg_zero_dur = float(np.mean(dur[zero_mask])) if n_zero > 0 else 0.0

    avg_chases_pos = float(np.mean(chs[pos_mask])) if n_pos > 0 else 0.0
    avg_chases_neg = float(np.mean(chs[neg_mask])) if n_neg > 0 else 0.0
    avg_chases_zero = float(np.mean(chs[zero_mask])) if n_zero > 0 else 0.0

    # Регрессия (смесь по эмпирическим долям)
    base_average_pnl_pct = p_pos * avg_pos_pnl + p_neg * avg_neg_pnl + p_zero * avg_zero_pnl
    base_average_position_duration_s = (
        p_pos * avg_pos_dur + p_neg * avg_neg_dur + p_zero * avg_zero_dur
    )

    # Безопасность от деления на 0
    seconds_in_year = 365 * 24 * 60 * 60.0
    positions_per_year = (seconds_in_year / base_average_position_duration_s) if base_average_position_duration_s > 0 else 0.0
    cost_per_year_pct = base_average_pnl_pct * positions_per_year
    cost_per_year_usd = cost_per_year_pct * float(volume_usd) / 100.0

    base_avg_chases_per_position = (
        p_pos * avg_chases_pos + p_neg * avg_chases_neg + p_zero * avg_chases_zero
    )
    base_avg_turnover_per_usd = positions_per_year * base_avg_chases_per_position
    base_avg_turnover_per_year = base_avg_turnover_per_usd * float(volume_usd)

    metrics['average_pnl_pct'] = base_average_pnl_pct
    metrics['average_position_duration_s'] = base_average_position_duration_s
    metrics['positions_per_year'] = positions_per_year
    metrics['cost_per_year_pct'] = cost_per_year_pct
    metrics['cost_per_year_usd'] = cost_per_year_usd
    metrics['avg_chases_per_position'] = base_avg_chases_per_position
    metrics['avg_turnover_per_usd'] = base_avg_turnover_per_usd
    metrics['avg_turnover_per_year'] = base_avg_turnover_per_year

    # Доп. раздельные метрики
    metrics['avg_pos_pnl_pct'] = avg_pos_pnl
    metrics['avg_neg_pnl_pct'] = avg_neg_pnl
    metrics['positive_share'] = p_pos
    metrics['negative_share'] = p_neg
    metrics['zero_share'] = p_zero
    metrics['avg_pos_duration_s'] = avg_pos_dur
    metrics['avg_neg_duration_s'] = avg_neg_dur
    metrics['avg_zero_duration_s'] = avg_zero_dur
    metrics['avg_chases_pos'] = avg_chases_pos
    metrics['avg_chases_neg'] = avg_chases_neg
    metrics['avg_chases_zero'] = avg_chases_zero

    return metrics

def base_report(test_data, symbol, conf):
    """
    Концепция базовых показателей: идет расчет стоимости полного покрытия
    (по времени) длительностью 1 год. Расчет является регрессией на основе
    данных истории и бэктеста.
    """

    # Checks
    assert isinstance(conf['test_volume_usd'], list), 'non-discrete test_volume_usd not supported'
    assert isinstance(conf.get('d_top_target_perc'), list), 'non-discrete d_top_target_perc not supported'
    assert isinstance(conf.get('d_btm_target_perc'), list), 'non-discrete d_btm_target_perc not supported'
    assert bool(conf.get('targets_sync', False)) is True, 'base_report requires targets_sync=true'

    # Gather params
    # only_side -> test_volume_usd -> target_perc -> list[(pnl_pct, exec_time, n_chases)]
    all_params = {}

    for item in test_data:
        total_trades_executed = int(item['stat']['agg']['total_trades_executed'])
        if total_trades_executed == 0:
            continue

        # Parametrization
        test_volume_usd = int(item['run_conf']['test_volume_usd'])
        only_side = int(item['run_conf']['only_side'])

        # Targets (synced)
        d_top = float(item['run_conf']['d_top_target_perc'])
        d_btm = float(item['run_conf']['d_btm_target_perc'])
        assert d_top == d_btm, 'targets_sync violated: d_top_target_perc != d_btm_target_perc'
        target_perc = int(d_top) if d_top.is_integer() else d_top

        # Metrics
        full_execution_time = float(item['stat']['agg']['full_execution_time'])
        pnl = float(item['stat']['agg']['pnl'])
        n_chases = int(item['stat']['agg']['n_chases'])

        # Add
        if only_side not in all_params:
            all_params[only_side] = {}

        if test_volume_usd not in all_params[only_side]:
            all_params[only_side][test_volume_usd] = {}

        if target_perc not in all_params[only_side][test_volume_usd]:
            all_params[only_side][test_volume_usd][target_perc] = []

        all_params[only_side][test_volume_usd][target_perc].append(
            (pnl / test_volume_usd * 100.0, full_execution_time, n_chases)
        )

    # Generate report
    csv = []

    for only_side in sorted(all_params):
        for test_volume_usd in sorted(all_params[only_side]):
            for target_perc in sorted(all_params[only_side][test_volume_usd]):
                # Convert to numpy array
                params = np.array(all_params[only_side][test_volume_usd][target_perc], dtype=np.float64)
                if len(params) == 0:
                    continue

                # Get metrics
                metrics = get_base_metrics(params, test_volume_usd)

                # Add to CSV
                csv.append({
                    'symbol': symbol,
                    'conf': conf['description'],
                    'only_side': only_side,
                    'test_volume_usd': test_volume_usd,
                    'target_perc': target_perc,
                    **metrics
                })

    return csv


In [4]:
# Generate report
import os, pandas as pd
from itables import show

DIR = '/_data/base'

# List all symbols
symbols = [fn for fn in os.listdir(DIR) if os.path.isdir(f'{DIR}/{fn}')]
symbols.sort()
print(f'Symbols: {symbols}')

csv = []

for symbol in symbols:
    print(f'Building reports for: {symbol}')

    # Load confs
    folder = f'/_data/base/{symbol}/results'
    confs = load_confs()
    print(f'Found {len(confs)} configs')

    for name, conf in confs.items():
        print(f'Building report for: {symbol} {name}')

        # Load tests
        test_data = load_tests(symbol, name)

        # Build report
        csv.extend(base_report(test_data, symbol, conf))

    break

# Convert to DataFrame
df = pd.DataFrame(csv)

# Show
show(df)

# Save
os.makedirs('reports', exist_ok=True)
df.to_csv('reports/base_report.csv', index=False)


Symbols: ['BTCUSDC', 'ETHUSDC', 'SOLUSDC']
Building reports for: BTCUSDC
Found 4 configs
Building report for: BTCUSDC list_01p


Loading tests for BTCUSDC list_01p: 100%|██████████| 50/50 [01:24<00:00,  1.68s/it]


Total size: 3762.107 MB
Loaded total: 4955419 tests from 50 files
Building report for: BTCUSDC list_01p_s


Loading tests for BTCUSDC list_01p_s: 100%|██████████| 50/50 [00:57<00:00,  1.16s/it]


Total size: 1335.975 MB
Loaded total: 4999771 tests from 50 files
Building report for: BTCUSDC list_1u_s


Loading tests for BTCUSDC list_1u_s: 100%|██████████| 50/50 [00:52<00:00,  1.05s/it]


Total size: 1335.819 MB
Loaded total: 4999772 tests from 50 files
Building report for: BTCUSDC list_1u


Loading tests for BTCUSDC list_1u: 100%|██████████| 50/50 [01:19<00:00,  1.59s/it]


Total size: 3770.879 MB
Loaded total: 4955399 tests from 50 files


Loading ITables v2.6.2 from the internet... (need help?)
